# R01_PipelineE_C2_LOSO_stability_shortlist.ipynb
### Shortlist = [3, 5, 9, 12]
### Output folder = /mnt/c/EEGFMRI/hmm/R01_rerun/02_derivatives/fusion_prep/fusion_hmm_LOSO_param_stability

## This Pipeline E gives you the “stability proof” for each shortlisted K:
### - state matchability (per-state corr; mean corr)
### - fold similarity matrices (signature, A)
### - A_std (transition stability)
### - matched FO + dwellA consistency

### That’s the exact evidence you’ll cite when choosing the final K and arguing the model reflects within-run dynamics rather than run identity.

In [ ]:
# =========================
# Cell 0 — USER INPUTS (edit here only)
# =========================
from pathlib import Path
import os

# -------------------------
# Shortlist for stability
# -------------------------
K_LIST = [3, 5, 9, 12]

# -------------------------
# Dataset identity
# -------------------------
DATA_VARIANT = "intermediate"
FEATURE_MODE = "nolags"   # "lags" or "nolags"
MINLEN       = 15

# -------------------------
# Paths (WSL format)
# -------------------------
FINAL_ROOT = Path("/mnt/c/EEGFMRI/hmm/R01_rerun/02_derivatives/fusion_prep/align_trmask_lags/FINAL_v3_gnorm_allTR") / DATA_VARIANT
MANIFEST_TSV = None  # auto-find if None

OUT_ROOT = Path("/mnt/c/EEGFMRI/hmm/R01_rerun/02_derivatives/fusion_prep/fusion_hmm_LOSO_param_stability") / f"C2_{DATA_VARIANT}_{FEATURE_MODE}_minlen{MINLEN}"

# -------------------------
# Feature layout (Pipeline B builds X = [BOLD | EEG])
# -------------------------
N_PARCELS = 200
TR_SEC    = 2.1

if FEATURE_MODE.lower() == "lags":
    LAGS_TR = [-1, 0, 1]
elif FEATURE_MODE.lower() == "nolags":
    LAGS_TR = [0]
else:
    raise ValueError("FEATURE_MODE must be 'lags' or 'nolags'")

D_BOLD  = N_PARCELS
D_EEG   = N_PARCELS * len(LAGS_TR)
D_TOTAL = D_BOLD + D_EEG

# -------------------------
# Windowing (C2 paradigm)
# -------------------------
SEQ_LEN    = 10
STEP_SIZE  = 1
BATCH_SIZE = 16

# IMPORTANT: shape-stable batching to prevent repeated recompiles / RAM creep
REBATCH_DROP_REMAINDER = True

# Shuffle buffer in "batches" after merging per-segment datasets
SHUFFLE_BUFFER = 2048

# -------------------------
# Fold-wise PCA (leakage-safe; fit on train only)
# -------------------------
N_BOLD_PCS = 40
N_EEG_PCS  = 40

# -------------------------
# HMM training hyperparameters
# -------------------------
LEARNING_RATE = 1e-3
N_EPOCHS      = 60
COV_EPS       = 1e-6
DIAGONAL_COVS = False   # recommended FULL covariances

# Initialization per seed
INIT_TAKE      = 0.30
INIT_EPOCHS    = 5
BIGK_THRESH    = 6
INIT_TAKE_BIGK = 0.20

# Multi-seed restarts + selection
SEEDS = [11, 23, 37, 53, 71]
VAL_SUBJECT_POLICY = "max_segments"

# Run-wise normalization within each split (train/val/test)
USE_RUNWISE_ZSCORE = True

# -------------------------
# FO-based anti-collapse screen (DO NOT gate on timepoint entropy)
# -------------------------
FO_MAX_THRESH           = 0.95   # collapse if one state dominates overall
FO_ACTIVE_THRESH        = 0.01   # state counts as "active" if FO > this
MIN_ACTIVE_STATES_BASE  = 3      # require at least min(3,K) active states

# -------------------------
# OOM hardening (WSL)
# -------------------------
# Disable XLA at import (must be set before TensorFlow is imported)
DISABLE_XLA_AT_IMPORT = True
if DISABLE_XLA_AT_IMPORT:
    os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"
    os.environ["TF_XLA_ENABLE_XLA_DEVICES"] = "0"
    os.environ["XLA_FLAGS"] = ""

FORCE_EAGER = False              # keep False for memory efficiency
DISABLE_PREFETCH  = True         # safer under WSL
DISABLE_CALLBACKS = True         # re-enable later if you want
GPU_MEMORY_LIMIT_MB = 4096       # None => memory_growth

# -------------------------
# Chunk + resume (recommended on WSL)
# -------------------------
RESUME_IF_EXISTS = True
MAX_NEW_FOLDS_PER_RUN = 2        # do a couple folds then restart kernel for safety

# Optional debug controls (leave as None normally)
DEBUG_MAX_FOLDS = None           # e.g., 1
DEBUG_SUBJECTS  = None           # e.g., ["sub-01","sub-02"]
DEBUG_SEEDS     = None           # e.g., [11]

print("K_LIST:", K_LIST)
print("OUT_ROOT:", OUT_ROOT)
print("FINAL_ROOT:", FINAL_ROOT)
print("SEQ_LEN/STEP/BATCH:", SEQ_LEN, STEP_SIZE, BATCH_SIZE)
print("REBATCH_DROP_REMAINDER:", REBATCH_DROP_REMAINDER)
print("SEEDS:", SEEDS)
print("MAX_NEW_FOLDS_PER_RUN:", MAX_NEW_FOLDS_PER_RUN)

# IMPORTANT NOTE:
# If you change DISABLE_XLA_AT_IMPORT, restart the kernel and run from Cell 0 again.

In [ ]:
# =========================
# Cell 1 — Imports + manifest resolution + load subject list
# =========================
import gc, math, json, re
from pathlib import Path

import numpy as np
import pandas as pd

import tensorflow as tf
from osl_dynamics.data import Data
from osl_dynamics.models.hmm import Config, Model

OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Save run meta for reproducibility
run_meta = dict(
    K_LIST=K_LIST,
    DATA_VARIANT=DATA_VARIANT,
    FEATURE_MODE=FEATURE_MODE,
    MINLEN=MINLEN,
    FINAL_ROOT=str(FINAL_ROOT),
    OUT_ROOT=str(OUT_ROOT),
    N_PARCELS=N_PARCELS,
    TR_SEC=TR_SEC,
    LAGS_TR=LAGS_TR,
    D_TOTAL=D_TOTAL,
    SEQ_LEN=SEQ_LEN,
    STEP_SIZE=STEP_SIZE,
    BATCH_SIZE=BATCH_SIZE,
    REBATCH_DROP_REMAINDER=REBATCH_DROP_REMAINDER,
    N_BOLD_PCS=N_BOLD_PCS,
    N_EEG_PCS=N_EEG_PCS,
    LEARNING_RATE=LEARNING_RATE,
    N_EPOCHS=N_EPOCHS,
    DIAGONAL_COVS=DIAGONAL_COVS,
    COV_EPS=COV_EPS,
    INIT_TAKE=INIT_TAKE,
    INIT_EPOCHS=INIT_EPOCHS,
    BIGK_THRESH=BIGK_THRESH,
    INIT_TAKE_BIGK=INIT_TAKE_BIGK,
    SEEDS=SEEDS,
    VAL_SUBJECT_POLICY=VAL_SUBJECT_POLICY,
    USE_RUNWISE_ZSCORE=USE_RUNWISE_ZSCORE,
    FO_MAX_THRESH=FO_MAX_THRESH,
    FO_ACTIVE_THRESH=FO_ACTIVE_THRESH,
    MIN_ACTIVE_STATES_BASE=MIN_ACTIVE_STATES_BASE,
    DISABLE_XLA_AT_IMPORT=DISABLE_XLA_AT_IMPORT,
    FORCE_EAGER=FORCE_EAGER,
    DISABLE_PREFETCH=DISABLE_PREFETCH,
    DISABLE_CALLBACKS=DISABLE_CALLBACKS,
    GPU_MEMORY_LIMIT_MB=GPU_MEMORY_LIMIT_MB,
)
(OUT_ROOT / "run_meta.json").write_text(json.dumps(run_meta, indent=2))

def auto_find_manifest(final_root: Path, feature_mode: str, minlen: int) -> Path:
    mode = feature_mode.lower()
    candidates = [
        final_root / f"hmm_segments_minlen{minlen}_{mode}" / "segments_manifest.tsv",
        final_root / f"hmm_segments_minlen{minlen}" / "segments_manifest.tsv",
    ]
    for m in candidates:
        if m.exists():
            return m
    hits = list(final_root.rglob("segments_manifest.tsv"))
    if hits:
        def score(p: Path):
            s = str(p)
            sc = 0
            if f"minlen{minlen}" in s: sc += 10
            if mode in s: sc += 5
            return sc
        hits = sorted(hits, key=score, reverse=True)
        return hits[0]
    raise FileNotFoundError(f"Could not find segments_manifest.tsv under {final_root}")

if MANIFEST_TSV is None:
    MANIFEST_TSV = auto_find_manifest(FINAL_ROOT, FEATURE_MODE, MINLEN)

print("MANIFEST_TSV:", MANIFEST_TSV)
manifest = pd.read_csv(MANIFEST_TSV, sep="\t")
print("Rows:", len(manifest))
print("Columns:", list(manifest.columns))
display(manifest.head())

if "run" not in manifest.columns or "seg_path" not in manifest.columns:
    raise ValueError("Expected manifest columns: 'run', 'seg_path'")

def parse_subject(run: str) -> str:
    parts = str(run).split("_")
    for p in parts:
        if p.startswith("sub-"):
            return p
    return parts[0]

manifest["subject"] = manifest["run"].apply(parse_subject)

# Sort deterministically
sort_cols = ["subject", "run"]
if "seg_id" in manifest.columns:
    sort_cols.append("seg_id")
manifest = manifest.sort_values(sort_cols).reset_index(drop=True)

SEG_ROOT = MANIFEST_TSV.parent
def resolve_seg_path(p: str) -> Path:
    pp = Path(p)
    return pp if pp.is_absolute() else (SEG_ROOT / pp)

manifest["seg_abs"] = [resolve_seg_path(p) for p in manifest["seg_path"].tolist()]
missing = manifest.loc[~manifest["seg_abs"].apply(lambda p: p.exists())]
if len(missing):
    display(missing.head(10))
    raise FileNotFoundError("Missing seg files referenced by manifest.")

subjects = sorted(manifest["subject"].unique().tolist())
if DEBUG_SUBJECTS is not None:
    subjects = [s for s in subjects if s in list(DEBUG_SUBJECTS)]

print("n_subjects:", len(subjects))
print(subjects)

def fold_dir(K, heldout):
    d = OUT_ROOT / f"K{K:02d}" / f"fold_holdout-{heldout}"
    d.mkdir(parents=True, exist_ok=True)
    return d

In [ ]:
# =========================
# Cell 2 — TF GPU config (cap 4GB) + sanity
# =========================
try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except Exception:
    pass

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    try:
        if GPU_MEMORY_LIMIT_MB is not None:
            tf.config.set_logical_device_configuration(
                gpus[0],
                [tf.config.LogicalDeviceConfiguration(memory_limit=int(GPU_MEMORY_LIMIT_MB))]
            )
            print("[INFO] GPU memory capped:", GPU_MEMORY_LIMIT_MB, "MB")
        else:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print("[INFO] memory_growth enabled")
    except Exception as e:
        print("[WARN] GPU config:", e)
else:
    print("[INFO] CPU-only")

try:
    tf.config.optimizer.set_jit(False)
except Exception:
    pass

if FORCE_EAGER:
    tf.config.run_functions_eagerly(True)
    print("[INFO] FORCE_EAGER=True")
else:
    print("[INFO] FORCE_EAGER=False")

_ = tf.matmul(tf.random.normal((64,64)), tf.random.normal((64,64)))
print("TF OK")

In [ ]:
# =========================
# Cell 3 — Helpers (C2 recipe + signatures + collapse screen)
# =========================
from scipy.optimize import linear_sum_assignment

def gc_now():
    gc.collect()

def stable_clear():
    # Light teardown (WSL-safe)
    try:
        tf.keras.backend.clear_session()
    except Exception:
        pass
    gc_now()

def load_segments(df: pd.DataFrame):
    # Lazy load for RAM safety
    Xs = []
    for p in df["seg_abs"].tolist():
        x = np.load(p).astype(np.float32)
        if x.ndim != 2 or x.shape[1] != D_TOTAL:
            raise ValueError(f"Bad segment shape {x.shape} for {p}")
        Xs.append(x)
    return Xs

def count_windows(X_list):
    n = 0
    for x in X_list:
        T = int(x.shape[0])
        if T >= SEQ_LEN:
            n += 1 + (T - SEQ_LEN) // STEP_SIZE
    return int(n)

def steps_from_windows(n_windows: int):
    if n_windows <= 0:
        return 0
    if REBATCH_DROP_REMAINDER:
        # floor because drop_remainder=True
        return int(n_windows // int(BATCH_SIZE))
    return int(math.ceil(n_windows / float(BATCH_SIZE)))

def runwise_zscore_segments(X_list, run_ids, sl: slice):
    # compute mean/std per run_id within this split only
    run_ids = np.asarray(run_ids, dtype=object)
    uniq = pd.unique(run_ids)

    mu, sd = {}, {}
    for r in uniq:
        idx = np.where(run_ids == r)[0]
        X = np.concatenate([X_list[i][:, sl] for i in idx], axis=0)
        m = X.mean(axis=0)
        s = X.std(axis=0, ddof=0)
        s[s < 1e-12] = 1.0
        mu[r] = m
        sd[r] = s

    out = []
    for X, r in zip(X_list, run_ids):
        Z = X.copy()
        Z[:, sl] = (Z[:, sl] - mu[r]) / sd[r]
        out.append(Z.astype(np.float32))
    return out

def fit_standardizer(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=0)
    sd = np.where(sd < 1e-12, 1.0, sd)
    return mu.astype(np.float32), sd.astype(np.float32)

def apply_standardizer(X, mu, sd):
    return ((X - mu) / sd).astype(np.float32)

def fit_pca(X, n_fixed):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
    n_comp = int(min(n_fixed, Vt.shape[0]))
    V = Vt[:n_comp].T.astype(np.float32)
    var = (S**2) / max(Xc.shape[0] - 1, 1)
    pve = float(np.sum(var[:n_comp]) / np.sum(var)) if np.sum(var) > 0 else np.nan
    return mu.ravel().astype(np.float32), V, pve

def apply_pca(X, mu, V):
    return ((X - mu) @ V).astype(np.float32)

def make_fold_preproc(X_tr_raw_list):
    Xtr = np.concatenate(X_tr_raw_list, axis=0)
    Xb, Xe = Xtr[:, :D_BOLD], Xtr[:, D_BOLD:]

    mu_b, sd_b = fit_standardizer(Xb)
    mu_e, sd_e = fit_standardizer(Xe)

    Xb_z = apply_standardizer(Xb, mu_b, sd_b)
    Xe_z = apply_standardizer(Xe, mu_e, sd_e)

    mu_pb, Vb, pve_b = fit_pca(Xb_z, N_BOLD_PCS)
    mu_pe, Ve, pve_e = fit_pca(Xe_z, N_EEG_PCS)

    params = dict(mu_b=mu_b, sd_b=sd_b, mu_e=mu_e, sd_e=sd_e,
                  mu_pb=mu_pb, Vb=Vb, mu_pe=mu_pe, Ve=Ve)
    meta = dict(pve_bold=pve_b, pve_eeg=pve_e,
                n_bold_pcs=int(Vb.shape[1]), n_eeg_pcs=int(Ve.shape[1]),
                D_pca=int(Vb.shape[1] + Ve.shape[1]))
    return params, meta

def apply_fold_preproc(X, params):
    Xb = apply_standardizer(X[:, :D_BOLD], params["mu_b"], params["sd_b"])
    Xe = apply_standardizer(X[:, D_BOLD:], params["mu_e"], params["sd_e"])
    Xb_p = apply_pca(Xb, params["mu_pb"], params["Vb"])
    Xe_p = apply_pca(Xe, params["mu_pe"], params["Ve"])
    return np.concatenate([Xb_p, Xe_p], axis=1).astype(np.float32)

def make_config(K, D):
    cfg = Config(
        n_states=K,
        n_channels=D,
        sequence_length=SEQ_LEN,
        learn_means=True,
        learn_covariances=True,
        learn_trans_prob=True,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        n_epochs=N_EPOCHS,
        covariances_epsilon=COV_EPS,
    )
    try:
        cfg.covariance_matrix_type = "diag" if DIAGONAL_COVS else "full"
    except Exception:
        pass
    # one init per seed (we do multi-seed restarts)
    try:
        cfg.n_init = 1
    except Exception:
        pass
    return cfg

def callbacks():
    if DISABLE_CALLBACKS:
        return []
    return []

def as_tf_dataset(data: Data, shuffle: bool):
    # concatenate=False => can return list of datasets; we merge them safely
    ds_obj = data.dataset(
        sequence_length=SEQ_LEN,
        step_size=STEP_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        concatenate=False,
    )
    if isinstance(ds_obj, (list, tuple)):
        ds_list = list(ds_obj)
        if len(ds_list) == 0:
            raise ValueError("Empty dataset list: no windows. Check SEQ_LEN/STEP_SIZE.")
        ds = ds_list[0]
        for d in ds_list[1:]:
            ds = ds.concatenate(d)
    else:
        ds = ds_obj

    # Shape-stabilize batches (prevents retracing/XLA recompiles)
    if REBATCH_DROP_REMAINDER:
        try:
            ds = ds.unbatch().batch(int(BATCH_SIZE), drop_remainder=True)
        except Exception:
            pass

    if shuffle:
        try:
            ds = ds.shuffle(buffer_size=int(SHUFFLE_BUFFER), reshuffle_each_iteration=True)
        except Exception:
            pass

    if not DISABLE_PREFETCH:
        try:
            ds = ds.prefetch(tf.data.AUTOTUNE)
        except Exception:
            pass

    return ds

def free_energy(model, data: Data):
    ds = as_tf_dataset(data, shuffle=False)
    fe = model.free_energy(ds)
    if isinstance(fe, (list, tuple, np.ndarray)):
        fe = float(np.asarray(fe).ravel()[0])
    return float(fe)

def normalize_alpha_list(alpha_like):
    # normalize to list of (T,K)
    if isinstance(alpha_like, (list, tuple)):
        out = []
        for a in alpha_like:
            a = np.asarray(a)
            if a.ndim == 2:
                out.append(a)
            elif a.ndim == 3:
                out.extend([a[i] for i in range(a.shape[0])])
            else:
                raise ValueError(f"Unexpected alpha element shape {a.shape}")
        return out
    a = np.asarray(alpha_like)
    if a.ndim == 2:
        return [a]
    if a.ndim == 3:
        return [a[i] for i in range(a.shape[0])]
    raise ValueError(f"Unexpected alpha shape {a.shape}")

def get_alpha_list(model, data: Data):
    if hasattr(model, "get_alpha"):
        return normalize_alpha_list(model.get_alpha(data, concatenate=False, verbose=0))
    if hasattr(model, "get_gamma"):
        return normalize_alpha_list(model.get_gamma(data, concatenate=False, verbose=0))
    raise AttributeError("Model lacks get_alpha/get_gamma")

def summarize_alpha(alpha_list, K, eps=1e-12):
    alpha_list = normalize_alpha_list(alpha_list)
    tot_T = 0
    fo_num = np.zeros(K, dtype=np.float64)
    ent_sum_norm = 0.0
    dwell_lengths = [[] for _ in range(K)]

    for a in alpha_list:
        a = np.asarray(a, dtype=np.float64)
        tot_T += a.shape[0]
        fo_num += a.sum(axis=0)

        a_clip = np.clip(a, eps, 1.0)
        Ht = -(a_clip * np.log(a_clip)).sum(axis=1)
        ent_sum_norm += (Ht / np.log(K)).sum()

        s = np.argmax(a, axis=1)
        if len(s) > 0:
            cur = s[0]
            run = 1
            for t in range(1, len(s)):
                if s[t] == cur:
                    run += 1
                else:
                    dwell_lengths[cur].append(run)
                    cur = s[t]
                    run = 1
            dwell_lengths[cur].append(run)

    fo = (fo_num / max(tot_T, 1))
    fo_max = float(np.max(fo)) if fo.size else np.nan
    ent_norm = float(ent_sum_norm / max(tot_T, 1))
    n_active = int(np.sum(fo > FO_ACTIVE_THRESH)) if fo.size else 0
    dwell_map_mean = np.array([np.mean(d) if len(d) else np.nan for d in dwell_lengths], dtype=np.float32)

    return fo.astype(np.float32), fo_max, ent_norm, n_active, dwell_map_mean.astype(np.float32), int(tot_T)

def fo_entropy_and_neff(fo, K, eps=1e-12):
    fo = np.asarray(fo, dtype=np.float64)
    fo = np.clip(fo, eps, 1.0)
    fo = fo / fo.sum()
    H = float(-(fo * np.log(fo)).sum())
    fo_entropy_norm = float(H / np.log(K))
    neff = float(np.exp(H))
    return fo_entropy_norm, neff

def is_collapsed(fo_max, n_active, K):
    min_active = min(int(MIN_ACTIVE_STATES_BASE), int(K))
    if not np.isfinite(fo_max):
        return True
    if fo_max > FO_MAX_THRESH:
        return True
    if int(n_active) < int(min_active):
        return True
    return False

def choose_best_candidate(cands):
    non = [c for c in cands if not c["collapsed"]]
    pool = non if len(non) else cands
    return sorted(pool, key=lambda c: (c["fe_val"], c["fo_max"], -c["n_active"], -c["fo_entropy"]))[0]

def dwell_from_A(A, eps=1e-12):
    A = np.asarray(A, dtype=np.float64)
    Akk = np.clip(np.diag(A), 0.0, 1.0 - eps)
    return (1.0 / (1.0 - Akk)).astype(np.float32)

def cov_to_corr_ut(C, eps=1e-12):
    C = np.asarray(C, dtype=np.float64)
    d = np.sqrt(np.clip(np.diag(C), eps, None))
    corr = C / (d[:, None] * d[None, :])
    iu = np.triu_indices(corr.shape[0], k=1)
    return corr[iu].astype(np.float32)

def ensure_cov_3d(covs):
    covs = np.asarray(covs)
    if covs.ndim == 2:
        # diagonal
        K, P = covs.shape
        out = np.zeros((K, P, P), dtype=covs.dtype)
        for k in range(K):
            np.fill_diagonal(out[k], covs[k])
        return out
    return covs

def backproject_cov_bold(covs_pca, Vb, nbpc):
    covs_pca = ensure_cov_3d(covs_pca)
    K = covs_pca.shape[0]
    out = []
    for k in range(K):
        Cbb = covs_pca[k, :nbpc, :nbpc]
        out.append((Vb @ Cbb @ Vb.T).astype(np.float32))
    return np.stack(out, axis=0)

def compute_signature_ut_bold(covs_pca, Vb, nbpc):
    cov_bold = backproject_cov_bold(covs_pca, Vb, nbpc)  # (K, 200, 200)
    sig_ut = np.stack([cov_to_corr_ut(cov_bold[k]) for k in range(cov_bold.shape[0])], axis=0)
    return sig_ut.astype(np.float32)

def match_states_to_reference(sig, ref_sig):
    """
    sig/ref_sig: (K, E) signature vectors
    returns:
      inv: index array length K where inv[j] = row index in 'sig' assigned to reference state j
      per_state_corr: correlation for each matched state (reference order)
    """
    K = sig.shape[0]
    # corr matrix: rows=fold states, cols=ref states
    corr = np.corrcoef(sig, ref_sig)[:K, K:]

    # maximize corr => minimize -corr
    row_ind, col_ind = linear_sum_assignment(-corr)

    inv = np.zeros(K, dtype=int)
    for r, c in zip(row_ind, col_ind):
        inv[c] = r

    per_state_corr = np.array([corr[inv[j], j] for j in range(K)], dtype=float)
    return inv, per_state_corr

In [ ]:
# =========================
# Cell 4 — Train LOSO folds for shortlisted K (resume + chunk safe)
# =========================
new_folds_done = 0

for K in K_LIST:
    print(f"\n==================== K={K} ====================")
    (OUT_ROOT / f"K{K:02d}").mkdir(parents=True, exist_ok=True)

    for fi, heldout in enumerate(subjects):
        if DEBUG_MAX_FOLDS is not None and fi >= int(DEBUG_MAX_FOLDS):
            break

        outdir = fold_dir(K, heldout)
        sentinel = outdir / "state_signature_corr_ut_bold.npy"
        if RESUME_IF_EXISTS and sentinel.exists():
            print("[SKIP]", heldout, "exists")
            continue

        # LOSO split
        test_df  = manifest.loc[manifest["subject"] == heldout].copy()
        train_df = manifest.loc[manifest["subject"] != heldout].copy()

        # inner validation subject selection
        if VAL_SUBJECT_POLICY == "max_segments":
            val_sub = train_df.groupby("subject").size().sort_values(ascending=False).index[0]
        else:
            val_sub = sorted(train_df["subject"].unique().tolist())[0]

        val_df = train_df.loc[train_df["subject"] == val_sub].copy()
        trn_df = train_df.loc[train_df["subject"] != val_sub].copy()

        # load raw segments lazily
        X_trn_raw = load_segments(trn_df)
        X_val_raw = load_segments(val_df)
        X_tst_raw = load_segments(test_df)

        trn_runs = trn_df["run"].tolist()
        val_runs = val_df["run"].tolist()
        tst_runs = test_df["run"].tolist()

        # run-wise zscore within each split
        if USE_RUNWISE_ZSCORE:
            X_trn_raw = runwise_zscore_segments(X_trn_raw, trn_runs, slice(0, D_BOLD))
            X_trn_raw = runwise_zscore_segments(X_trn_raw, trn_runs, slice(D_BOLD, D_TOTAL))
            X_val_raw = runwise_zscore_segments(X_val_raw, val_runs, slice(0, D_BOLD))
            X_val_raw = runwise_zscore_segments(X_val_raw, val_runs, slice(D_BOLD, D_TOTAL))
            X_tst_raw = runwise_zscore_segments(X_tst_raw, tst_runs, slice(0, D_BOLD))
            X_tst_raw = runwise_zscore_segments(X_tst_raw, tst_runs, slice(D_BOLD, D_TOTAL))

        # leakage-safe fold preproc (fit on training only)
        params, meta = make_fold_preproc(X_trn_raw)

        X_trn = [apply_fold_preproc(x, params) for x in X_trn_raw]
        X_val = [apply_fold_preproc(x, params) for x in X_val_raw]
        X_tst = [apply_fold_preproc(x, params) for x in X_tst_raw]

        # explicit steps
        nwin_tr = count_windows(X_trn)
        nwin_va = count_windows(X_val)
        steps_tr = steps_from_windows(nwin_tr)
        steps_va = steps_from_windows(nwin_va)

        if steps_tr == 0 or steps_va == 0:
            # fallback if drop_remainder makes steps 0 (rare)
            print("[WARN] steps==0 under drop_remainder; disabling rebatch for this fold.")
            globals()["REBATCH_DROP_REMAINDER"] = False
            steps_tr = steps_from_windows(nwin_tr)
            steps_va = steps_from_windows(nwin_va)

        train_data = Data(X_trn)
        val_data   = Data(X_val)
        test_data  = Data(X_tst)

        cfg = make_config(K, meta["D_pca"])
        init_take = INIT_TAKE_BIGK if K >= BIGK_THRESH else INIT_TAKE
        seeds_run = DEBUG_SEEDS if DEBUG_SEEDS is not None else SEEDS

        print(
            f"--- fold={fi+1}/{len(subjects)} holdout={heldout} | val={val_sub} | "
            f"train_segs={len(X_trn)} val_segs={len(X_val)} test_segs={len(X_tst)} | "
            f"D_pca={meta['D_pca']} | steps={steps_tr} ---"
        )

        # Multi-seed candidate pass (val FE + FO metrics)
        candidates = []
        for seed in seeds_run:
            stable_clear()
            np.random.seed(int(seed))
            tf.random.set_seed(int(seed))

            train_ds = as_tf_dataset(train_data, shuffle=True)
            val_ds   = as_tf_dataset(val_data, shuffle=False)

            m = Model(cfg)

            # init
            try:
                try:
                    m.random_subset_initialization(train_data, take=float(init_take), n_epochs=int(INIT_EPOCHS), n_init=1)
                except TypeError:
                    m.random_subset_initialization(train_data, take=float(init_take), n_epochs=int(INIT_EPOCHS))
            except Exception as e:
                continue

            # fit
            try:
                try:
                    m.fit(
                        train_ds,
                        validation_data=val_ds,
                        steps_per_epoch=int(steps_tr),
                        validation_steps=int(steps_va),
                        callbacks=callbacks(),
                    )
                except TypeError:
                    m.fit(
                        train_ds,
                        validation_data=val_ds,
                        steps_per_epoch=int(steps_tr),
                        validation_steps=int(steps_va),
                    )
            except Exception as e:
                continue

            fe_val = free_energy(m, val_data)
            alpha_val = get_alpha_list(m, val_data)
            fo, fo_max, ent_norm, n_active, _, _ = summarize_alpha(alpha_val, K)
            fo_entropy, neff = fo_entropy_and_neff(fo, K)
            collapsed = is_collapsed(fo_max, n_active, K)

            candidates.append(dict(
                seed=int(seed),
                fe_val=float(fe_val),
                fo_max=float(fo_max),
                entropy_norm=float(ent_norm),
                n_active=int(n_active),
                fo_entropy=float(fo_entropy),
                neff=float(neff),
                collapsed=bool(collapsed),
            ))

            # cleanup per seed
            try:
                del m, train_ds, val_ds, alpha_val
            except Exception:
                pass
            gc_now()

        if len(candidates) == 0:
            raise RuntimeError(f"No candidates trained for fold holdout={heldout}, K={K}")

        best = choose_best_candidate(candidates)

        # Refit best seed ONCE and save artifacts
        stable_clear()
        np.random.seed(int(best["seed"]))
        tf.random.set_seed(int(best["seed"]))

        train_ds = as_tf_dataset(train_data, shuffle=True)
        val_ds   = as_tf_dataset(val_data, shuffle=False)

        m = Model(cfg)
        try:
            try:
                m.random_subset_initialization(train_data, take=float(init_take), n_epochs=int(INIT_EPOCHS), n_init=1)
            except TypeError:
                m.random_subset_initialization(train_data, take=float(init_take), n_epochs=int(INIT_EPOCHS))

            try:
                m.fit(
                    train_ds,
                    validation_data=val_ds,
                    steps_per_epoch=int(steps_tr),
                    validation_steps=int(steps_va),
                    callbacks=callbacks(),
                )
            except TypeError:
                m.fit(
                    train_ds,
                    validation_data=val_ds,
                    steps_per_epoch=int(steps_tr),
                    validation_steps=int(steps_va),
                )
        except Exception as e:
            raise

        # Extract model params
        A = np.asarray(m.get_trans_prob(), dtype=np.float32)
        covs = ensure_cov_3d(np.asarray(m.get_covariances(), dtype=np.float32))

        np.save(outdir / "trans_prob.npy", A)
        np.save(outdir / "covs_pca.npy", covs)

        # Means are optional depending on model
        try:
            means = np.asarray(m.get_means(), dtype=np.float32)
            np.save(outdir / "means_pca.npy", means)
        except Exception:
            pass

        # Signature for matching across folds
        sig_ut = compute_signature_ut_bold(covs, params["Vb"], N_BOLD_PCS)
        np.save(outdir / "state_signature_corr_ut_bold.npy", sig_ut)

        # Train/test FE
        fe_train = float(free_energy(m, train_data))
        fe_test  = float(free_energy(m, test_data))

        # Test summaries
        alpha_test = get_alpha_list(m, test_data)
        fo_t, fo_max_t, ent_t, n_active_t, dwell_map_mean_t, totT_test = summarize_alpha(alpha_test, K)
        fo_entropy_t, neff_t = fo_entropy_and_neff(fo_t, K)
        dwellA = dwell_from_A(A)

        fold_summ = dict(
            K=int(K),
            heldout_subject=str(heldout),
            val_subject=str(val_sub),
            total_T_test=int(totT_test),
            FO=fo_t.tolist(),
            FO_max=float(fo_max_t),
            entropy_mean_norm=float(ent_t),
            n_active=int(n_active_t),
            fo_entropy=float(fo_entropy_t),
            neff=float(neff_t),
            dwell_map_mean_TR=dwell_map_mean_t.tolist(),
            dwell_A_TR=dwellA.tolist(),
            dwell_A_sec=(dwellA * TR_SEC).tolist(),
            fe_train=float(fe_train),
            fe_test=float(fe_test),
        )
        (outdir / "fold_summaries.json").write_text(json.dumps(fold_summ, indent=2))

        fold_info = dict(
            K=int(K),
            heldout_subject=str(heldout),
            val_subject=str(val_sub),
            seq_len=int(SEQ_LEN),
            step_size=int(STEP_SIZE),
            batch_size=int(BATCH_SIZE),
            n_train_segments=int(len(X_trn)),
            n_val_segments=int(len(X_val)),
            n_test_segments=int(len(X_tst)),
            nwin_train=int(nwin_tr),
            nwin_val=int(nwin_va),
            steps_per_epoch=int(steps_tr),
            validation_steps=int(steps_va),
            NBPC=int(N_BOLD_PCS),
            NEPC=int(N_EEG_PCS),
            D_pca=int(meta["D_pca"]),
            pve_bold=float(meta["pve_bold"]),
            pve_eeg=float(meta["pve_eeg"]),
            seeds=list(map(int, seeds_run)),
            init_take=float(init_take),
            best_candidate=best,
            candidates=candidates,
            runwise_zscore=bool(USE_RUNWISE_ZSCORE),
            rebatch_drop_remainder=bool(REBATCH_DROP_REMAINDER),
            xla_disabled_at_import=bool(DISABLE_XLA_AT_IMPORT),
        )
        (outdir / "fold_info.json").write_text(json.dumps(fold_info, indent=2))
        (outdir / "best_candidate.json").write_text(json.dumps(best, indent=2))
        (outdir / "candidates_index.json").write_text(json.dumps(candidates, indent=2))

        np.savez_compressed(
            outdir / "preproc_params.npz",
            mu_b=params["mu_b"], sd_b=params["sd_b"],
            mu_e=params["mu_e"], sd_e=params["sd_e"],
            mu_pb=params["mu_pb"], Vb=params["Vb"],
            mu_pe=params["mu_pe"], Ve=params["Ve"],
        )

        # Cleanup fold
        try:
            del m, train_ds, val_ds, alpha_test
        except Exception:
            pass
        stable_clear()

        new_folds_done += 1
        if MAX_NEW_FOLDS_PER_RUN is not None and new_folds_done >= int(MAX_NEW_FOLDS_PER_RUN):
            raise SystemExit("Chunk complete. Restart kernel and run Cell 4 again to resume.")

In [ ]:
# =========================
# Cell 5 — Stability analysis per K (run AFTER training completes for each K)
# =========================
import matplotlib.pyplot as plt

def plot_matrix(M, title, out_png, xlabels=None, ylabels=None):
    plt.figure(figsize=(6,5))
    plt.imshow(M, aspect="auto")
    plt.title(title)
    plt.colorbar()
    if xlabels is not None:
        plt.xticks(np.arange(len(xlabels)), xlabels)
    if ylabels is not None:
        plt.yticks(np.arange(len(ylabels)), ylabels)
    plt.tight_layout()
    plt.savefig(out_png, dpi=220)
    plt.close()

def stability_for_K(K: int):
    K_out = OUT_ROOT / f"K{K:02d}"
    folds = sorted([p for p in K_out.glob("fold_holdout-*") if p.is_dir()])
    if len(folds) == 0:
        raise RuntimeError(f"No folds found for K={K} under {K_out}")

    # Load signatures + A + fold summaries
    sigs, As, sums = [], [], []
    fold_names = []
    for fd in folds:
        sig_path = fd / "state_signature_corr_ut_bold.npy"
        A_path   = fd / "trans_prob.npy"
        s_path   = fd / "fold_summaries.json"
        if not (sig_path.exists() and A_path.exists() and s_path.exists()):
            raise FileNotFoundError(f"Missing fold artifacts in {fd}")
        sigs.append(np.load(sig_path))
        As.append(np.load(A_path))
        sums.append(json.loads(s_path.read_text()))
        fold_names.append(fd.name)

    ref_sig = sigs[0]
    matched_dir = K_out / "matched_folds"
    matched_dir.mkdir(exist_ok=True)

    sigs_m, As_m = [], []
    match_rows = []

    for fd, sig, A in zip(folds, sigs, As):
        inv, per_state_corr = match_states_to_reference(sig, ref_sig)

        sig_m = sig[inv, :]
        A_m = A[np.ix_(inv, inv)]

        outd = matched_dir / fd.name
        outd.mkdir(parents=True, exist_ok=True)
        np.save(outd / "match_reorder_idx.npy", inv.astype(int))
        np.save(outd / "match_per_state_corr.npy", per_state_corr.astype(np.float32))

        sigs_m.append(sig_m)
        As_m.append(A_m)

        row = {"fold": fd.name, "mean_state_corr": float(np.mean(per_state_corr))}
        for i, c in enumerate(per_state_corr, start=1):
            row[f"state_corr_s{i:02d}"] = float(c)
        match_rows.append(row)

    pd.DataFrame(match_rows).to_csv(K_out / "state_matching_scores.tsv", sep="\t", index=False)

    # Fold similarity matrices
    F = len(folds)
    sim_sig = np.zeros((F, F), dtype=float)
    sim_A   = np.zeros((F, F), dtype=float)

    def sim_sig_pair(i, j):
        # mean corr across matched states
        a = sigs_m[i]; b = sigs_m[j]
        cs = [np.corrcoef(a[k], b[k])[0,1] for k in range(a.shape[0])]
        return float(np.mean(cs))

    def sim_A_pair(i, j):
        a = As_m[i].ravel(); b = As_m[j].ravel()
        return float(np.corrcoef(a, b)[0,1])

    for i in range(F):
        for j in range(F):
            sim_sig[i, j] = sim_sig_pair(i, j)
            sim_A[i, j]   = sim_A_pair(i, j)

    fold_labels = [str(i) for i in range(1, F+1)]
    pd.DataFrame(sim_sig, index=fold_labels, columns=fold_labels).to_csv(K_out / "sim_matrix_signature.tsv", sep="\t")
    pd.DataFrame(sim_A,   index=fold_labels, columns=fold_labels).to_csv(K_out / "sim_matrix_A.tsv", sep="\t")

    # A mean/std
    A_stack = np.stack(As_m, axis=0)
    A_mean = A_stack.mean(axis=0)
    A_std  = A_stack.std(axis=0, ddof=1)

    state_labels = [str(i) for i in range(1, K+1)]
    pd.DataFrame(A_mean, index=state_labels, columns=state_labels).to_csv(K_out / "A_mean.tsv", sep="\t")
    pd.DataFrame(A_std,  index=state_labels, columns=state_labels).to_csv(K_out / "A_std.tsv", sep="\t")

    # Matched fold summaries table (FO + dwellA)
    rows = []
    for fd, summ in zip(folds, sums):
        inv = np.load(matched_dir / fd.name / "match_reorder_idx.npy").astype(int)
        fo = np.asarray(summ["FO"], dtype=float)[inv]
        dwellA = np.asarray(summ["dwell_A_TR"], dtype=float)[inv]

        r = dict(
            fold=fd.name,
            heldout_subject=summ.get("heldout_subject", ""),
            val_subject=summ.get("val_subject", ""),
            total_T_test=int(summ.get("total_T_test", -1)),
            FO_max=float(np.max(fo)),
            n_active=int(np.sum(fo > FO_ACTIVE_THRESH)),
            neff=float(summ.get("neff", np.nan)),
            fe_test=float(summ.get("fe_test", np.nan)),
        )
        for i, v in enumerate(fo, start=1):
            r[f"FO_s{i:02d}"] = float(v)
        for i, v in enumerate(dwellA, start=1):
            r[f"dwellA_TR_s{i:02d}"] = float(v)
        rows.append(r)

    pd.DataFrame(rows).to_csv(K_out / "fold_summaries_table_matched.tsv", sep="\t", index=False)

    # Plots
    plot_matrix(sim_sig, f"K={K} Fold Similarity (Signature)", K_out / "plot_sim_signature.png", fold_labels, fold_labels)
    plot_matrix(sim_A,   f"K={K} Fold Similarity (A)",         K_out / "plot_sim_A.png", fold_labels, fold_labels)
    plot_matrix(A_std,   f"K={K} Transition Std (A_std)",      K_out / "plot_A_std.png", state_labels, state_labels)

    print(f"[K={K}] stability outputs written to {K_out}")

# Run stability analysis for all K (only after training is finished for those K)
for K in K_LIST:
    stability_for_K(int(K))